# Softmax + Cross-Entropy Gradient

Companion notebook for the wiki page: [Softmax + Cross-Entropy Gradient](https://ml-viz.vercel.app/wiki/softmax-cross-entropy)

We implement softmax and cross-entropy from scratch, verify the clean gradient formula $\partial L/\partial z_k = \hat{p}_k - \mathbf{1}[k=y]$ with finite differences, and visualise the loss surface.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive. Changes to this view are not saved.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

plt.style.use('dark_background')

## From-scratch implementation

In [ ]:
def softmax(z):
    """Numerically stable softmax (subtract max before exp)."""
    e = np.exp(z - z.max())
    return e / e.sum()

def cross_entropy_loss(z, y):
    """Cross-entropy loss for true class y."""
    return -np.log(softmax(z)[y] + 1e-12)

def softmax_ce_grad(z, y):
    """Exact gradient: dL/dz_k = p_hat_k - 1[k==y]."""
    grad = softmax(z).copy()
    grad[y] -= 1
    return grad

## Reproduce the 3-class worked example

In [ ]:
z = np.array([2.0, 1.0, -1.0])
y = 0  # true class

p_hat = softmax(z)
loss  = cross_entropy_loss(z, y)
grad  = softmax_ce_grad(z, y)

print(f"Logits z:       {z}")
print(f"Softmax p̂:     {p_hat.round(4)}")
print(f"Loss:           {loss:.4f}   (= -log({p_hat[y]:.4f}))")
print(f"Exact gradient: {grad.round(4)}")
print(f"  correct class k=0: {grad[0]:.4f}  = {p_hat[0]:.4f} - 1")
print(f"  wrong   class k=1: {grad[1]:.4f}  = {p_hat[1]:.4f}")
print(f"  wrong   class k=2: {grad[2]:.4f}  = {p_hat[2]:.4f}")
print(f"  gradient sums to:  {grad.sum():.6f}  (should be 0)")

## Finite-difference verification

In [ ]:
eps = 1e-5
fd = np.zeros(len(z))
for i in range(len(z)):
    z_p, z_m = z.copy(), z.copy()
    z_p[i] += eps; z_m[i] -= eps
    fd[i] = (cross_entropy_loss(z_p, y) - cross_entropy_loss(z_m, y)) / (2 * eps)

print(f"Finite-diff grad: {fd.round(6)}")
print(f"Exact grad:       {grad.round(6)}")
print(f"Max abs error:    {np.abs(grad - fd).max():.2e}")
print(f"Gradients match:  {np.allclose(grad, fd, atol=1e-7)}")

## Visualise: how loss varies with the correct-class logit

In [ ]:
z_sweep = np.linspace(-3, 5, 200)
losses = [cross_entropy_loss(np.array([z0, 1.0, -1.0]), 0) for z0 in z_sweep]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(z_sweep, losses, color='#6366f1', lw=2)
ax1.axvline(2.0, color='#f59e0b', ls='--', label='z₀ = 2 (worked example)')
ax1.set_xlabel('z₀ (correct-class logit)')
ax1.set_ylabel('Cross-entropy loss')
ax1.set_title('Loss vs correct-class logit')
ax1.legend(); ax1.grid(True, alpha=0.2)

# Gradient visualisation
grads_correct = [softmax_ce_grad(np.array([z0, 1.0, -1.0]), 0)[0] for z0 in z_sweep]
ax2.plot(z_sweep, grads_correct, color='#22d3ee', lw=2, label='∂L/∂z₀ = p̂₀ − 1')
ax2.axhline(0, color='#444', lw=0.8)
ax2.axvline(2.0, color='#f59e0b', ls='--', label='z₀ = 2')
ax2.set_xlabel('z₀')
ax2.set_ylabel('Gradient')
ax2.set_title('Gradient of correct-class logit')
ax2.legend(); ax2.grid(True, alpha=0.2)

plt.tight_layout()
plt.suptitle('Softmax + Cross-Entropy: loss and gradient landscape', y=1.02)
plt.show()

## Log-sum-exp numerical stability

In [ ]:
# Demonstrate overflow without the max-subtraction trick
z_large = np.array([1000.0, 999.0, 998.0])

print("Direct exp (no stability trick):")
print(f"  np.exp(1000) = {np.exp(1000.0)}")  # inf

print("\nWith max subtraction:")
e = np.exp(z_large - z_large.max())
p = e / e.sum()
print(f"  softmax({z_large}) = {p.round(4)}")
print(f"  sum = {p.sum():.6f}  (correct)")

## ✏️ Your turn

**Exercise 1:** Extend `softmax_ce_grad` to handle a **batch** of examples. Input `Z` is `(N, K)`, `y` is `(N,)`. Return the mean gradient over the batch.

**Exercise 2:** Implement **temperature-scaled softmax** `softmax(z, T)` where dividing by temperature $T$ before softmax controls sharpness. Verify: at $T \to 0$ it approaches argmax; at $T \to \infty$ it approaches uniform.

**Exercise 3:** Show that for binary classification ($K=2$), softmax + cross-entropy is equivalent to sigmoid + binary cross-entropy. Verify numerically.

In [ ]:
# Exercise 1: batch gradient
def batch_softmax_ce_grad(Z, y):
    """
    Z: (N, K) logits
    y: (N,)  true class indices
    Returns: (K,) mean gradient
    """
    # TODO(you): vectorise the single-example formula
    pass

# Test
Z_test = np.array([[2.0, 1.0, -1.0], [0.5, 2.0, 1.0]])
y_test = np.array([0, 1])
# assert batch_softmax_ce_grad(Z_test, y_test).shape == (3,)

In [ ]:
# Exercise 2: temperature-scaled softmax
def softmax_temp(z, T=1.0):
    # TODO(you): divide logits by T before softmax
    pass

# for T in [0.01, 0.5, 1.0, 5.0, 100.0]:
#     print(f"T={T:6.2f}: {softmax_temp(z, T).round(3)}")

<details>
<summary>Solution — Exercise 1</summary>

```python
def batch_softmax_ce_grad(Z, y):
    N, K = Z.shape
    # Compute softmax for each row
    e = np.exp(Z - Z.max(axis=1, keepdims=True))
    P = e / e.sum(axis=1, keepdims=True)  # (N, K)
    # Subtract 1 from the true-class column
    P[np.arange(N), y] -= 1
    return P.mean(axis=0)  # (K,)
```
</details>

<details>
<summary>Solution — Exercise 2</summary>

```python
def softmax_temp(z, T=1.0):
    z_scaled = z / T
    e = np.exp(z_scaled - z_scaled.max())
    return e / e.sum()
```
</details>